In [ ]:
!pip install -q datasets==3.6.0
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q transformers peft accelerate bitsandbytes
!pip install tree-sitter
!pip install tree-sitter-java

!pip install tree-sitter-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 667.5/667.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 6.2 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, PeftModel
from google.colab import userdata
import datasets
from datasets import load_dataset, Dataset
import torch
import pandas as pd
import tree_sitter
import ast
import json
from tree_sitter import Language, Parser
import tree_sitter_java
import tree_sitter_python
import subprocess
import tempfile
import os

In [ ]:
print(datasets.__version__)

3.6.0


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

HF_TOKEN = userdata.get('HF_TOKEN')
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "bigcode/starcoder2-3b"

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(model_name, token=HF_TOKEN, quantization_config=bnb_config).to(device)

config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/7.88k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/442k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 12.1GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
!unzip -o /content/starcoder2-python-java-custom-lora.zip -d /content/starcoder2-python-java-custom-lora/

Archive:  /content/starcoder2-python-java-custom-lora.zip
  inflating: /content/starcoder2-python-java-custom-lora/adapter_config.json  
  inflating: /content/starcoder2-python-java-custom-lora/tokenizer.json  
  inflating: /content/starcoder2-python-java-custom-lora/adapter_model.safetensors  
  inflating: /content/starcoder2-python-java-custom-lora/training_args.bin  
  inflating: /content/starcoder2-python-java-custom-lora/tokenizer_config.json  
  inflating: /content/starcoder2-python-java-custom-lora/README.md  


In [ ]:
# Load the LoRA adapters from the checkpoint
model_to_test = PeftModel.from_pretrained(model, "/content/starcoder2-python-java-custom-lora")

# Set the model to evaluation mode and move to device
model_to_test = model_to_test.eval().to(device)

print("Model loaded successfully for testing.")

Model loaded successfully for testing.


In [ ]:
#model_to_test = None

### Calculating CodeBLEU, CodeBERT, and ROUGE scores

In [ ]:
!pip install -q evaluate sentence-transformers nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00


In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=98cca6245116e5825b526d658255b892e636da5c5f58e609384bba31299b1fd8
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
import evaluate
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk

# Download necessary NLTK data for BLEU (if not already present)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# Load BLEU and ROUGE metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

# Load a SentenceTransformer model for CodeBERT-like embeddings
# Using a general-purpose model; for true CodeBERT, a code-specific model would be better if available via SentenceTransformers
# For better performance on code, consider models like 'sentence-transformers/all-distilroberta-v1' or a fine-tuned code model.
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def calculate_code_metrics(reference_code, generated_code):
    # BLEU score
    # The 'evaluate' library expects lists of strings for references and predictions
    # For a single pair, we wrap them in lists
    bleu_score = bleu.compute(predictions=[generated_code], references=[[reference_code]])

    # ROUGE score
    rouge_score = rouge.compute(predictions=[generated_code], references=[reference_code])

    # CodeBERT-like similarity using sentence embeddings
    # Generate embeddings for both original and generated code
    embeddings_ref = embedding_model.encode(reference_code, convert_to_tensor=True)
    embeddings_gen = embedding_model.encode(generated_code, convert_to_tensor=True)

    # Calculate cosine similarity
    # Reshape for sklearn's cosine_similarity if they are 1D tensors/arrays
    if embeddings_ref.dim() == 1:
        embeddings_ref = embeddings_ref.unsqueeze(0)
    if embeddings_gen.dim() == 1:
        embeddings_gen = embeddings_gen.unsqueeze(0)

    codebert_similarity = cosine_similarity(embeddings_ref.cpu(), embeddings_gen.cpu())[0][0]

    return {
        'bleu': bleu_score['bleu'],
        'rouge1': rouge_score['rouge1'],
        'rouge2': rouge_score['rouge2'],
        'rougeL': rouge_score['rougeL'],
        'codebert_similarity': float(codebert_similarity)
    }

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def extract_code(text):
    if "### Response" in text:
        return text.split("### Response")[-1].strip()

    return text.strip()

def run_mbpp_tests(code, test_list):
    namespace = {}

    try:
        exec(code, namespace)

        for test in test_list:
            exec(test, namespace)

        return True

    except Exception:
        return False

In [ ]:
JAVA_LANGUAGE = Language(
    tree_sitter_java.language()
)

parser = Parser()
parser.language = JAVA_LANGUAGE

In [ ]:
def extract_java_ir(code):

    tree = parser.parse(
        bytes(code, "utf8")
    )

    root = tree.root_node

    ir = {
        "functions": 0,
        "for_loops": 0,
        "while_loops": 0,
        "ifs": 0,
        "returns": 0,
        "assignments": 0,
        "binary_ops": [],
        "comparisons": [],
        "calls": [],
        "data_structures": [],
        "recursion": False
    }

    method_names = set()

    def walk(node):

        if node.type == "method_declaration":

            ir["functions"] += 1

            for child in node.children:

                if child.type == "identifier":
                    method_names.add(
                        child.text.decode()
                    )

        elif node.type in [
            "for_statement",
            "enhanced_for_statement"
        ]:
            ir["for_loops"] += 1

        elif node.type == "while_statement":
            ir["while_loops"] += 1

        elif node.type == "if_statement":
            ir["ifs"] += 1

        elif node.type == "return_statement":
            ir["returns"] += 1

        elif node.type == "assignment_expression":
            ir["assignments"] += 1

        elif node.type == "method_invocation":

            for child in node.children:

                if child.type == "identifier":

                    name = child.text.decode()

                    ir["calls"].append(name)

                    if name in method_names:
                        ir["recursion"] = True

                    break

        elif node.type in [
            "+",
            "-",
            "*",
            "/",
            "%",
            "==",
            "!=",
            "<",
            ">",
            "<=",
            ">="
        ]:
            ir["binary_ops"].append(
                node.type
            )

        elif node.type == "array_creation_expression":
            ir["data_structures"].append(
                "array"
            )

        for child in node.children:
            walk(child)

    walk(root)

    for key in [
        "binary_ops",
        "comparisons",
        "calls",
        "data_structures"
    ]:
        ir[key] = sorted(
            list(set(ir[key]))
        )

    return ir

In [ ]:
import ast

def extract_python_ir(code):

    tree = ast.parse(code)

    ir = {
        "functions": 0,
        "for_loops": 0,
        "while_loops": 0,
        "ifs": 0,
        "returns": 0,
        "assignments": 0,
        "binary_ops": [],
        "comparisons": [],
        "calls": [],
        "data_structures": [],
        "recursion": False
    }

    function_names = set()

    class Visitor(ast.NodeVisitor):

        def visit_FunctionDef(self, node):

            ir["functions"] += 1
            function_names.add(node.name)

            self.generic_visit(node)

        def visit_For(self, node):
            ir["for_loops"] += 1
            self.generic_visit(node)

        def visit_While(self, node):
            ir["while_loops"] += 1
            self.generic_visit(node)

        def visit_If(self, node):
            ir["ifs"] += 1
            self.generic_visit(node)

        def visit_Return(self, node):
            ir["returns"] += 1
            self.generic_visit(node)

        def visit_Assign(self, node):
            ir["assignments"] += 1
            self.generic_visit(node)

        def visit_Call(self, node):

            if isinstance(node.func, ast.Name):

                ir["calls"].append(node.func.id)

                if node.func.id in function_names:
                    ir["recursion"] = True

            self.generic_visit(node)

        def visit_BinOp(self, node):

            op = type(node.op).__name__

            ir["binary_ops"].append(op)

            self.generic_visit(node)

        def visit_Compare(self, node):

            for op in node.ops:
                ir["comparisons"].append(
                    type(op).__name__
                )

            self.generic_visit(node)

        def visit_List(self, node):
            ir["data_structures"].append("list")
            self.generic_visit(node)

        def visit_Dict(self, node):
            ir["data_structures"].append("dict")
            self.generic_visit(node)

        def visit_Set(self, node):
            ir["data_structures"].append("set")
            self.generic_visit(node)

        def visit_Tuple(self, node):
            ir["data_structures"].append("tuple")
            self.generic_visit(node)

    Visitor().visit(tree)

    for key in [
        "binary_ops",
        "comparisons",
        "calls",
        "data_structures"
    ]:
        ir[key] = sorted(list(set(ir[key])))

    return ir

In [ ]:
def similarity(ir1, ir2):

    score = 0
    total = 0

    numeric_fields = [
        "functions",
        "for_loops",
        "while_loops",
        "ifs",
        "returns",
        "assignments"
    ]

    for field in numeric_fields:

        total += 1

        if ir1[field] == ir2[field]:
            score += 1

    list_fields = [
        "binary_ops",
        "comparisons",
        "calls",
        "data_structures"
    ]

    for field in list_fields:

        total += 1

        s1 = set(ir1[field])
        s2 = set(ir2[field])

        union = s1.union(s2)

        if len(union) == 0:
            score += 1
        else:
            score += (
                len(s1.intersection(s2))
                / len(union)
            )

    total += 1

    if ir1["recursion"] == ir2["recursion"]:
        score += 1

    return round(score / total, 4)

In [ ]:
def compare_java_python(java_problem, python_code):
    java_ir = extract_java_ir(java_problem)
    python_ir = extract_python_ir(python_code)

    score = similarity(
        java_ir,
        python_ir
    )
    print(f"Similarity: {score}")
    return score

In [ ]:
def compare_java_java(java_code1, java_code2):
    java_ir1 = extract_java_ir(java_code1)
    java_ir2 = extract_java_ir(java_code2)

    score = similarity(
        java_ir1,
        java_ir2
    )
    # print(f"Similarity: {score}")
    return score

In [ ]:
def check_java_compilation(java_code):
    # Create a temporary directory
    with tempfile.TemporaryDirectory() as tmpdir:
        # Write the Java code to a .java file
        file_path = os.path.join(tmpdir, "Solution.java")
        with open(file_path, "w") as f:
            f.write(java_code)

        # Try to compile the Java file
        try:
            # Use subprocess to run javac
            compile_process = subprocess.run(
                ["javac", file_path],
                capture_output=True,
                text=True,
                check=False # Do not raise an exception for non-zero exit codes
            )
            # If javac returns 0, compilation was successful
            if compile_process.returncode == 0:
                return True
            else:
                # print(f"Compilation error: {compile_process.stderr}")
                return False
        except FileNotFoundError:
            print("Error: javac command not found. Make sure Java Development Kit (JDK) is installed and in your PATH.")
            return False
        except Exception as e:
            print(f"An unexpected error occurred during compilation check: {e}")
            return False

In [ ]:
translation_pairs_df = pd.read_csv('/content/java_python_translation_pairs_corrected_6.csv')
display(translation_pairs_df.head())

,qid,text,java_code,python_code,python_function,python_test_cases,java_test_cases,python_execution_status,java_execution_status
0,602,Find the first repeated character in a given s...,import java.io.*;\nimport java.lang.*;\nimport...,"def first_repeated_char(str1):\n for index,c ...",first_repeated_char,"[\n ""assert first_repeated_char('abcabc') == ...",public class TestRunner {\n private static ...,PASS,PASS
1,603,get a lucid number smaller than or equal to n.,import java.io.*;\nimport java.lang.*;\nimport...,def get_ludic(n):\n\tludics = []\n\tfor i in r...,get_ludic,"[\n ""assert get_ludic(10) == [1, 2, 3, 5, 7]""...",public class TestRunner {\n private static ...,PASS,PASS
2,604,reverse words in a given string.,import java.io.*;\nimport java.lang.*;\nimport...,def reverse_words(s):\n return ' '.join...,reverse_words,"[\n ""assert reverse_words('python program') =...",public class TestRunner {\n private static ...,PASS,PASS
3,605,check if the given integer is a prime number.,import java.io.*;\nimport java.lang.*;\nimport...,def prime_num(num):\n if num >=1:\n for i i...,prime_num,"[\n ""assert prime_num(13) == True"",\n ""asser...",public class TestRunner {\n private static ...,PASS,PASS
4,606,convert degrees to radians.,import java.io.*;\nimport java.lang.*;\nimport...,import math\ndef radian_degree(degree):\n radi...,radian_degree,"[\n ""assert radian_degree(90) == 1.5707963267...",public class TestRunner {\n private static ...,PASS,PASS


In [ ]:
def generate_python_from_java_solution_finetuned_model(model_eval, prompt,test_cases_str,  num_solutions=1):
    full_prompt = f"""
             ### Instruction
             {prompt}
             ### Test cases
             {test_cases_str}
             ### Response
             """

    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)

    outputs = model_eval.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.2,
        top_p=0.95,
        num_return_sequences=num_solutions,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

    return generated_texts

In [ ]:
results_data = []
k = 1 # For pass@K, generate k solutions per problem

for index, row in translation_pairs_df[200:].iterrows():
    print(f"processing question {row['qid']}")
    java_problem_text = f"""
    convert the following java code to python
    {row['java_code']}
    """
    java_problem = row['java_code']
    python_solution_original = row['python_code'] # Keep original Python solution
    # python_test_cases in translation_pairs_df is a string representation of a list of strings
    python_test_cases_list = ast.literal_eval(row['python_test_cases'])

    # Generate Python solutions (num_solutions = k for pass@k)
    generated_solutions = generate_python_from_java_solution_finetuned_model(model_to_test, java_problem_text, python_test_cases_list, num_solutions=k)

    all_passes_for_qid = False
    generated_python_code_first_attempt = ''

    if generated_solutions:
        generated_python_code_first_attempt = extract_code(generated_solutions[0])
        for generated_python_code_full_response in generated_solutions:
            current_generated_python_code = extract_code(generated_python_code_full_response)

            # Check if any of the generated solutions pass all tests
            if run_mbpp_tests(current_generated_python_code, python_test_cases_list):
                all_passes_for_qid = True
                break # Found a passing solution for this qid

    # Calculate AST similarity score
    ast_score = 0.0
    if generated_python_code_first_attempt:
        try:
            ast_score = compare_java_python(java_problem, generated_python_code_first_attempt)
        except Exception as e:
            print(f"Error calculating AST similarity for qid {row['qid']}: {e}")

    # Calculate other code metrics (BLEU, ROUGE, CodeBERT-like)
    code_metrics = {
        'bleu': 0.0,
        'rouge1': 0.0,
        'rouge2': 0.0,
        'rougeL': 0.0,
        'codebert_similarity': 0.0
    }
    if generated_python_code_first_attempt:
        try:
            code_metrics = calculate_code_metrics(python_solution_original, generated_python_code_first_attempt)
        except Exception as e:
            print(f"Error calculating other metrics for qid {row['qid']}: {e}")

    # Collect all data for the new DataFrame
    results_data.append({
        'qid': row['qid'],
        'text': row['text'], # Include original 'text' column if desired
        'java_code': java_problem,
        'python_code_original': python_solution_original,
        'generated_python_code': generated_python_code_first_attempt,
        'ast_similarity_score': ast_score,
        'passes_all_tests': all_passes_for_qid,
        'bleu_score': code_metrics['bleu'],
        'rouge1_score': code_metrics['rouge1'],
        'rouge2_score': code_metrics['rouge2'],
        'rougeL_score': code_metrics['rougeL'],
        'codebert_similarity_score': code_metrics['codebert_similarity']
    })

results_df = pd.DataFrame(results_data)

print("Processing complete. Displaying the new DataFrame with results:")
display(results_df.head())

Similarity: 0.2727
processing question 953


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Similarity: 0.6364
processing question 954


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Similarity: 0.8182
processing question 955


...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Similarity: 0.8182
processing question 974


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Similarity: 0.5455
Processing complete. Displaying the new DataFrame with results:


,qid,text,java_code,python_code_original,generated_python_code,ast_similarity_score,passes_all_tests,bleu_score,rouge1_score,rouge2_score,rougeL_score,codebert_similarity_score
0,827,sum a specific column of a list in a given li...,import java.io.*;\nimport java.lang.*;\nimport...,"def sum_column(list1, C):\n result = sum(ro...","def sum_column(list1, c):\n sum = 0\n for i ...",0.6364,True,0.215341,0.647059,0.250000,0.588235,0.903839
1,828,"count alphabets,digits and special charactes ...",import java.io.*;\nimport java.lang.*;\nimport...,def count_alpha_dig_spl(string):\n alphabets=...,def count_alpha_dig_spl(string):\n alpha = ...,0.6364,True,0.563070,0.773333,0.657534,0.773333,0.942451
2,829,find out the second most repeated (or frequen...,import java.io.*;\nimport java.lang.*;\nimport...,from collections import Counter \n\t\ndef seco...,def second_frequent(input):\n freq = {}\n fo...,0.4545,True,0.249686,0.441176,0.121212,0.264706,0.889121
3,830,round up a number to specific digits.,import java.io.*;\nimport java.lang.*;\nimport...,"import math\ndef round_up(a, digits):\n n =...","def round_up(a, digits):\n return round(a, di...",0.5455,False,0.270351,0.666667,0.480000,0.666667,0.913347
4,831,Count equal element pairs from the given list.,import java.io.*;\nimport java.lang.*;\nimport...,"def count_Pairs(arr,n): \n cnt = 0; \n f...","def count_Pairs(arr, n): \n\tcount = 0\n\tfor ...",0.6364,True,0.764247,0.877193,0.763636,0.877193,0.978087


In [ ]:
pass_at_k = (results_df['passes_all_tests'].sum() / len(results_df)) * 100
print(f"Pass@{k}: {pass_at_k:.2f}%")

mean_ast_similarity = results_df['ast_similarity_score'].mean()
mean_bleu = results_df['bleu_score'].mean()
mean_rouge1 = results_df['rouge1_score'].mean()
mean_rouge2 = results_df['rouge2_score'].mean()
mean_rougeL = results_df['rougeL_score'].mean()
mean_codebert_similarity = results_df['codebert_similarity_score'].mean()

print(f"Mean AST Similarity Score: {mean_ast_similarity:.4f}")
print(f"Mean BLEU Score: {mean_bleu:.4f}")
print(f"Mean ROUGE-1 Score: {mean_rouge1:.4f}")
print(f"Mean ROUGE-2 Score: {mean_rouge2:.4f}")
print(f"Mean ROUGE-L Score: {mean_rougeL:.4f}")
print(f"Mean CodeBERT Similarity Score: {mean_codebert_similarity:.4f}")


Pass@1: 60.61%
Mean AST Similarity Score: 0.5827
Mean BLEU Score: 0.3676
Mean ROUGE-1 Score: 0.6511
Mean ROUGE-2 Score: 0.4525
Mean ROUGE-L Score: 0.5919
Mean CodeBERT Similarity Score: 0.8927


In [ ]:
composite_score = (
    0.35 * results_df["codebert_similarity_score"] +
    0.25 * results_df["ast_similarity_score"] +
    0.15 * results_df["rougeL_score"] +
    0.10 * results_df["rouge1_score"] +
    0.05 * results_df["rouge2_score"] +
    0.10 * results_df["bleu_score"]
)
results_df['composite_score'] = composite_score
print("Composite translation score calculated. Displaying updated DataFrame head:")
display(results_df.head())

Composite translation score calculated. Displaying updated DataFrame head:


,qid,text,java_code,python_code_original,generated_python_code,ast_similarity_score,passes_all_tests,bleu_score,rouge1_score,rouge2_score,rougeL_score,codebert_similarity_score,composite_score
0,827,sum a specific column of a list in a given li...,import java.io.*;\nimport java.lang.*;\nimport...,"def sum_column(list1, C):\n result = sum(ro...","def sum_column(list1, c):\n sum = 0\n for i ...",0.6364,True,0.215341,0.647059,0.250000,0.588235,0.903839,0.662419
1,828,"count alphabets,digits and special charactes ...",import java.io.*;\nimport java.lang.*;\nimport...,def count_alpha_dig_spl(string):\n alphabets=...,def count_alpha_dig_spl(string):\n alpha = ...,0.6364,True,0.563070,0.773333,0.657534,0.773333,0.942451,0.771475
2,829,find out the second most repeated (or frequen...,import java.io.*;\nimport java.lang.*;\nimport...,from collections import Counter \n\t\ndef seco...,def second_frequent(input):\n freq = {}\n fo...,0.4545,True,0.249686,0.441176,0.121212,0.264706,0.889121,0.539670
3,830,round up a number to specific digits.,import java.io.*;\nimport java.lang.*;\nimport...,"import math\ndef round_up(a, digits):\n n =...","def round_up(a, digits):\n return round(a, di...",0.5455,False,0.270351,0.666667,0.480000,0.666667,0.913347,0.673748
4,831,Count equal element pairs from the given list.,import java.io.*;\nimport java.lang.*;\nimport...,"def count_Pairs(arr,n): \n cnt = 0; \n f...","def count_Pairs(arr, n): \n\tcount = 0\n\tfor ...",0.6364,True,0.764247,0.877193,0.763636,0.877193,0.978087,0.835335


In [ ]:
mean_composite_score = results_df['composite_score'].mean()
print(f"Mean Composite Translation Score: {mean_composite_score:.4f}")

Mean Composite Translation Score: 0.6714


In [ ]:
results_df.to_csv('finetuned_java_to_python_translation_results_200_332.csv', index=False)
print("DataFrame saved to 'finetuned_java_to_python_translation_results.csv'")

DataFrame saved to 'finetuned_java_to_python_translation_results.csv'


### Python to Java Translation and Metric Calculation

First, let's define a function to generate Java code from a Python problem description using our fine-tuned model.

In [ ]:
def generate_java_from_python_solution_finetuned_model(model_eval, python_code_problem, test_cases_str, num_solutions=1):
    full_prompt = f"""
             ### Instruction
             Convert the following Python code to Java:
             {python_code_problem}
             ### Test cases
             {test_cases_str}
             ### Response
             """

    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)

    outputs = model_eval.generate(
        **inputs,
        max_new_tokens=768,
        do_sample=True,
        temperature=0.2,
        top_p=0.95,
        num_return_sequences=num_solutions,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

    return generated_texts

Next, we'll need a function to compare two Java codes using their Abstract Syntax Trees (ASTs).

Now, let's run the evaluation loop for Python to Java translation, calculate all specified metrics, and store them in a new DataFrame.

In [ ]:
finetuned_model_python_to_java_results_data = []
k=1 # For Pass@k

for index, row in translation_pairs_df[200:].iterrows():
    print(f"Processing qid: {row['qid']}")
    python_problem = row['python_code']
    java_solution_original = row['java_code'] # Reference Java solution
    test_cases_str = row['java_test_cases'] # Extract test cases

    # Check if original Java solution compiles (still useful to know)
    original_compiles = check_java_compilation(java_solution_original)
    # Check if original Java solution passes all tests
    original_passes_all_tests = run_java_tests(java_solution_original, test_cases_str)

    # Generate Java solutions (num_solutions = k for pass@k) using the base model
    generated_solutions_list = generate_java_from_python_solution_finetuned_model(model_to_test, python_problem, test_cases_str, num_solutions=k)

    passes_all_tests = False # This will be our pass@k
    compiles_successfully = False # Track if any generated solution compiles
    first_generated_java_code = ''

    if generated_solutions_list:
        first_generated_java_code = extract_code(generated_solutions_list[0])
        for generated_java_code_full_response in generated_solutions_list:
            current_generated_java_code = extract_code(generated_java_code_full_response)
            # Check if any of the generated solutions compile
            if check_java_compilation(current_generated_java_code):
                compiles_successfully = True
            # Check if any of the generated solutions pass all tests
            if run_java_tests(current_generated_java_code, test_cases_str):
                passes_all_tests = True

            if compiles_successfully and passes_all_tests: # If both conditions met, no need to check further for this qid
                break

    # Calculate AST similarity score between the *first* generated Java and original Java
    ast_score = 0.0
    if generated_solutions_list:
        try:
            # first_generated_java_code is already extracted above
            ast_score = compare_java_java(java_solution_original, first_generated_java_code)
        except Exception as e:
            print(f"Error calculating AST similarity for qid {row['qid']}: {e}")

    # Calculate other code metrics (BLEU, ROUGE, CodeBERT-like) between the *first* generated Java and original Java
    code_metrics = {
        'bleu': 0.0,
        'rouge1': 0.0,
        'rouge2': 0.0,
        'rougeL': 0.0,
        'codebert_similarity': 0.0
    }
    if generated_solutions_list:
        try:
            # first_generated_java_code is already extracted above
            code_metrics = calculate_code_metrics(java_solution_original, first_generated_java_code)
        except Exception as e:
            print(f"Error calculating other metrics for qid {row['qid']}: {e}")


    finetuned_model_python_to_java_results_data.append({
        'qid': row['qid'],
        'text': row['text'],
        'python_code_original': python_problem,
        'java_code_original': java_solution_original,
        'generated_java_code_first_attempt': first_generated_java_code if generated_solutions_list else '',
        'original_compiles': original_compiles,
        'original_passes_all_tests': original_passes_all_tests, # New field
        'compiles_successfully': compiles_successfully, # Re-added field
        'passes_all_tests': passes_all_tests, # Renamed and updated logic
        'ast_similarity_score': ast_score,
        'bleu_score': code_metrics['bleu'],
        'rouge1_score': code_metrics['rouge1'],
        'rouge2_score': code_metrics['rouge2'],
        'rougeL_score': code_metrics['rougeL'],
        'codebert_similarity_score': code_metrics['codebert_similarity']
    })

finetuned_model_python_to_java_results_df = pd.DataFrame(finetuned_model_python_to_java_results_data)

print("Processing complete for Base Model Python-to-Java translation. Displaying results:")
display(finetuned_model_python_to_java_results_df.head())

Processing qid: 827


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Processing qid: 828


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Processing qid: 829


...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Processing qid: 934


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Processing qid: 935


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
pass_at_k_python_to_java_finetuned = (finetuned_model_python_to_java_results_df['passes_all_tests'].sum() / len(finetuned_model_python_to_java_results_df)) * 100
print(f"Finetuned Model Pass@{k} (Generated Java): {pass_at_k_python_to_java_finetuned:.2f}%")

original_pass_at_k_python_to_java_finetuned = (finetuned_model_python_to_java_results_df['original_passes_all_tests'].sum() / len(finetuned_model_python_to_java_results_df)) * 100
print(f"Finetuned Model Pass@{k} (Original Java): {original_pass_at_k_python_to_java_finetuned:.2f}%")

compilation_rate_finetuned = (finetuned_model_python_to_java_results_df['compiles_successfully'].sum() / len(finetuned_model_python_to_java_results_df)) * 100
print(f"Finetuned Model Compilation Rate (Generated Java): {compilation_rate_finetuned:.2f}%")

original_compilation_rate_finetuned = (finetuned_model_python_to_java_results_df['original_compiles'].sum() / len(finetuned_model_python_to_java_results_df)) * 100
print(f"Finetuned Model Compilation Rate (Original Java): {original_compilation_rate_finetuned:.2f}%")

mean_ast_similarity_python_to_java_finetuned = finetuned_model_python_to_java_results_df['ast_similarity_score'].mean()
mean_bleu_python_to_java_finetuned = finetuned_model_python_to_java_results_df['bleu_score'].mean()
mean_rouge1_python_to_java_finetuned = finetuned_model_python_to_java_results_df['rouge1_score'].mean()
mean_rouge2_python_to_java_results_df = finetuned_model_python_to_java_results_df['rouge2_score'].mean()
mean_rougeL_python_to_java_finetuned = finetuned_model_python_to_java_results_df['rougeL_score'].mean()
mean_codebert_similarity_python_to_java_finetuned = finetuned_model_python_to_java_results_df['codebert_similarity_score'].mean()

print(f"Mean AST Similarity Score (Finetuned Model Python-to-Java): {mean_ast_similarity_python_to_java_finetuned:.4f}")
print(f"Mean BLEU Score (Finetuned Model Python-to-Java): {mean_bleu_python_to_java_finetuned:.4f}")
print(f"Mean ROUGE-1 Score (Finetuned Model Python-to-Java): {mean_rouge1_python_to_java_finetuned:.4f}")
print(f"Mean ROUGE-2 Score (Finetuned Model Python-to-Java): {mean_rouge2_python_to_java_results_df:.4f}")
print(f"Mean ROUGE-L Score (Finetuned Model Python-to-Java): {mean_rougeL_python_to_java_finetuned:.4f}")
print(f"Mean CodeBERT Similarity Score (Finetuned Model Python-to-Java): {mean_codebert_similarity_python_to_java_finetuned:.4f}")

Finetuned Model Pass@1 (Generated Java): 54.00%
Finetuned Model Pass@1 (Original Java): 99.00%
Finetuned Model Compilation Rate (Generated Java): 93.00%
Finetuned Model Compilation Rate (Original Java): 100.00%
Mean AST Similarity Score (Finetuned Model Python-to-Java): 0.7683
Mean BLEU Score (Finetuned Model Python-to-Java): 0.6794
Mean ROUGE-1 Score (Finetuned Model Python-to-Java): 0.7839
Mean ROUGE-2 Score (Finetuned Model Python-to-Java): 0.6777
Mean ROUGE-L Score (Finetuned Model Python-to-Java): 0.7554
Mean CodeBERT Similarity Score (Finetuned Model Python-to-Java): 0.9183


### Composite Translation Score for Python to Java

In [ ]:
finetuned_model_python_to_java_results_df["translation_score"] = (
    0.35 * finetuned_model_python_to_java_results_df["codebert_similarity_score"] +
    0.25 * finetuned_model_python_to_java_results_df["ast_similarity_score"] +
    0.15 * finetuned_model_python_to_java_results_df["rougeL_score"] +
    0.10 * finetuned_model_python_to_java_results_df["rouge1_score"] +
    0.05 * finetuned_model_python_to_java_results_df["rouge2_score"] +
    0.10 * finetuned_model_python_to_java_results_df["bleu_score"]
)

print("Composite translation score calculated for Python to Java. Displaying updated DataFrame head:")
display(finetuned_model_python_to_java_results_df.head(10))

Composite translation score calculated for Python to Java. Displaying updated DataFrame head:


,qid,text,python_code_original,java_code_original,generated_java_code_first_attempt,original_compiles,original_passes_all_tests,compiles_successfully,passes_all_tests,ast_similarity_score,bleu_score,rouge1_score,rouge2_score,rougeL_score,codebert_similarity_score,translation_score
0,713,check if the given tuple contains all valid v...,def check_valid(test_tup):\n res = not any(ma...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,True,True,True,0.7045,0.681030,0.800000,0.630952,0.741176,0.910409,0.785595
1,714,Count the number of distinct power of prime fa...,def count_Fac(n): \n m = n \n count = 0...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,True,True,True,0.8182,0.883308,0.905983,0.834783,0.905983,0.847516,0.857746
2,715,convert the given string of integers into a t...,def str_to_tuple(test_str):\n res = tuple(map...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,True,True,False,0.8545,0.797235,0.860335,0.757062,0.849162,0.963567,0.881858
3,716,find the perimeter of a rombus.,def rombus_perimeter(a):\n perimeter=4*a\n r...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,True,True,True,0.7727,0.507278,0.782609,0.743363,0.782609,0.943321,0.806885
4,717,calculate the standard deviation.,import math\nimport sys\ndef sd_calc(data):\n ...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,True,True,False,1.0000,0.642971,0.743961,0.604878,0.676329,0.930383,0.846021
5,719,Write a function that matches a string that ha...,import re\ndef text_match(text):\n patt...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,True,True,False,0.8636,0.713141,0.761194,0.636364,0.716418,0.875998,0.809214
6,721,find a path with the maximum average over all...,"M = 100\ndef maxAverageOfPath(cost, N): \n\tdp...",import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,True,True,False,0.7825,0.627522,0.770642,0.621538,0.691131,0.960437,0.806341
7,723,count the same pair in two given lists using ...,from operator import eq\ndef count_same_pair(n...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,True,True,False,0.7273,0.851951,0.910448,0.857143,0.902985,0.947424,0.867968
8,724,calculate the sum of all digits of the base t...,"def power_base_sum(base, power):\n return s...",import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,True,True,False,0.8788,0.580434,0.794702,0.657718,0.768212,0.919838,0.827275
9,725,extract values between quotation marks of the...,import re\ndef extract_quotation(text1):\n re...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,True,True,False,0.6318,0.772060,0.764398,0.708995,0.764398,0.954475,0.795771


In [ ]:
composite_score_python_to_java_finetuned = finetuned_model_python_to_java_results_df['translation_score'].mean()
print(f"Mean Composite Translation Score (Finetuned Model Python-to-Java): {composite_score_python_to_java_finetuned:.4f}")

Mean Composite Translation Score (Finetuned Model Python-to-Java): 0.8070


In [ ]:
finetuned_model_python_to_java_results_df.to_csv('finetuned_python_to_java_translation_results_100_200.csv', index=False)
print("DataFrame saved to 'finetuned_python_to_java_translation_results.csv'")

DataFrame saved to 'finetuned_python_to_java_translation_results.csv'


To determine if the generated code passes the tests, we need a function to compile and run the generated Java code along with the provided test runner code. We'll extract the class name from the generated code and then compile both the solution and the `TestRunner` class.

In [ ]:
import re

def get_java_class_name(java_code):
    # Regex to find 'class ClassName {'
    match = re.search(r'class\s+(\w+)\s*{', java_code)
    if match:
        return match.group(1)
    return "Solution" # Default if not found

Next, we need a function to compile and run the generated Java code against a set of test cases and check if all tests pass. This function will extend the `check_java_compilation` logic to also execute the compiled code.

In [ ]:
def run_java_tests(generated_solution_code, test_runner_code_str):
    with tempfile.TemporaryDirectory() as tmpdir:
        # Extract the class name from the generated solution code
        solution_class_name = get_java_class_name(generated_solution_code)

        # Write the generated solution code
        solution_file_path = os.path.join(tmpdir, f"{solution_class_name}.java")
        with open(solution_file_path, "w") as f:
            f.write(generated_solution_code)

        # Write the test runner code
        test_runner_file_path = os.path.join(tmpdir, "TestRunner.java")
        with open(test_runner_file_path, "w") as f:
            f.write(test_runner_code_str)

        # Compile both Java files
        compile_process = subprocess.run(
            ["javac", solution_file_path, test_runner_file_path],
            capture_output=True,
            text=True,
            check=False,
            timeout=10 # Add timeout for compilation
        )

        if compile_process.returncode != 0:
            # Compilation failed
            # print(f"Compilation failed:\n{compile_process.stderr}")
            return False

        # If compilation is successful, run the TestRunner
        try:
            run_process = subprocess.run(
                ["java", "-classpath", tmpdir, "TestRunner"],
                capture_output=True,
                text=True,
                timeout=10, # Timeout for test execution
                check=False
            )

            if run_process.returncode != 0:
                # Runtime error or tests failed (e.g., AssertionError)
                # print(f"Test execution failed or runtime error:\n{run_process.stderr}")
                return False

            # Check for a success message in stdout
            if "All Java tests passed" in run_process.stdout:
                return True
            else:
                # Tests failed, but no explicit error in stderr (e.g. assertEquals failed)
                # print(f"Tests did not pass. Output:\n{run_process.stdout}")
                return False

        except subprocess.TimeoutExpired:
            # print("Test execution timed out")
            return False
        except Exception as e:
            # print(f"An unexpected error occurred during test execution: {e}")
            return False

Now, let's define a new function `generate_java_from_text_solution` that generates Java code directly from a text description. This will ensure we are evaluating 'text to Java generation' as requested.

In [ ]:
def generate_java_from_text_solution(prompt_text, test_cases, num_solutions=1):
    full_prompt = f"""
             ### Instruction
             Generate Java code that solves the following problem:
             {prompt_text}

             ### Test cases
             {test_cases}

             ### Response
             """

    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)

    outputs = model_to_test.generate(
        **inputs,
        max_new_tokens=768,
        do_sample=True,
        temperature=0.2,
        top_p=0.95,
        num_return_sequences=num_solutions,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

    return generated_texts

Finally, we will run the evaluation loop, generating Java code from the text descriptions, executing the provided test cases, and calculating the pass@k metric.

In [ ]:
java_text_gen_results_data = []
k = 1 # For pass@1

for index, row in test_cases_df.iterrows():
    problem_description = row['text']
    java_solution_original = row['java_code'] # Reference Java solution
    test_cases_str = row['java_test_cases']

    # Generate Java solutions (num_solutions = k for pass@k)
    generated_solutions_list = generate_java_from_text_solution(problem_description, test_cases_str, num_solutions=k)

    all_passes_for_qid = False
    for generated_java_code_full_response in generated_solutions_list:
        generated_java_code = extract_code(generated_java_code_full_response)

        # Check if any of the generated solutions pass all tests using the provided test runner code
        if run_java_tests(generated_java_code, test_cases_str):
            all_passes_for_qid = True
            break # Found a passing solution for this qid

    # Calculate AST similarity score between the *first* generated Java and original Java
    # Only if there was at least one generated solution.
    ast_score = 0.0
    if generated_solutions_list:
        try:
            first_generated_java_code = extract_code(generated_solutions_list[0])
            ast_score = compare_java_java(java_solution_original, first_generated_java_code)
        except Exception as e:
            print(f"Error calculating AST similarity for qid {row['qid']}: {e}")

    # Calculate other code metrics (BLEU, ROUGE, CodeBERT-like) between the *first* generated Java and original Java
    code_metrics = {
        'bleu': 0.0,
        'rouge1': 0.0,
        'rouge2': 0.0,
        'rougeL': 0.0,
        'codebert_similarity': 0.0
    }
    if generated_solutions_list:
        try:
            first_generated_java_code = extract_code(generated_solutions_list[0])
            code_metrics = calculate_code_metrics(java_solution_original, first_generated_java_code)
        except Exception as e:
            print(f"Error calculating other metrics for qid {row['qid']}: {e}")


    java_text_gen_results_data.append({
        'qid': row['qid'],
        'text': row['text'],
        'java_code_original': java_solution_original,
        'generated_java_code_first_attempt': extract_code(generated_solutions_list[0]) if generated_solutions_list else '',
        'passes_all_tests': all_passes_for_qid,
        'ast_similarity_score': ast_score,
        'bleu_score': code_metrics['bleu'],
        'rouge1_score': code_metrics['rouge1'],
        'rouge2_score': code_metrics['rouge2'],
        'rougeL_score': code_metrics['rougeL'],
        'codebert_similarity_score': code_metrics['codebert_similarity']
    })

java_text_gen_results_df = pd.DataFrame(java_text_gen_results_data)

print("Processing complete. Displaying the new DataFrame with Text-to-Java generation results:")
display(java_text_gen_results_df.head())

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

Processing complete. Displaying the new DataFrame with Text-to-Java generation results:


,qid,text,java_code_original,generated_java_code_first_attempt,passes_all_tests,ast_similarity_score,bleu_score,rouge1_score,rouge2_score,rougeL_score,codebert_similarity_score
0,602,Find the first repeated character in a given s...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.6000,0.584753,0.689655,0.573427,0.675862,0.950750
1,603,get a lucid number smaller than or equal to n.,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,False,0.3131,0.726709,0.737288,0.675214,0.737288,0.914966
2,604,reverse words in a given string.,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.9091,0.920185,0.924242,0.846154,0.924242,0.986594
3,605,check if the given integer is a prime number.,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.4886,0.728459,0.809524,0.709677,0.746032,0.929562
4,606,convert degrees to radians.,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.8636,0.823912,0.903226,0.835165,0.881720,0.910040


### Pass@K and Mean Scores for Text to Java Generation

In [ ]:
pass_at_k = (java_text_gen_results_df['passes_all_tests'].sum() / len(java_text_gen_results_df)) * 100
print(f"Pass@{k}: {pass_at_k:.2f}%")

mean_ast_similarity_text_gen = java_text_gen_results_df['ast_similarity_score'].mean()
mean_bleu_text_gen = java_text_gen_results_df['bleu_score'].mean()
mean_rouge1_text_gen = java_text_gen_results_df['rouge1_score'].mean()
mean_rouge2_text_gen = java_text_gen_results_df['rouge2_score'].mean()
mean_rougeL_text_gen = java_text_gen_results_df['rougeL_score'].mean()
mean_codebert_similarity_text_gen = java_text_gen_results_df['codebert_similarity_score'].mean()

print(f"Mean AST Similarity Score (Text-to-Java): {mean_ast_similarity_text_gen:.4f}")
print(f"Mean BLEU Score (Text-to-Java): {mean_bleu_text_gen:.4f}")
print(f"Mean ROUGE-1 Score (Text-to-Java): {mean_rouge1_text_gen:.4f}")
print(f"Mean ROUGE-2 Score (Text-to-Java): {mean_rouge2_text_gen:.4f}")
print(f"Mean ROUGE-L Score (Text-to-Java): {mean_rougeL_text_gen:.4f}")
print(f"Mean CodeBERT Similarity Score (Text-to-Java): {mean_codebert_similarity_text_gen:.4f}")

Pass@1: 49.70%
Mean AST Similarity Score (Text-to-Java): 0.7384
Mean BLEU Score (Text-to-Java): 0.6961
Mean ROUGE-1 Score (Text-to-Java): 0.8026
Mean ROUGE-2 Score (Text-to-Java): 0.7116
Mean ROUGE-L Score (Text-to-Java): 0.7813
Mean CodeBERT Similarity Score (Text-to-Java): 0.9485


### Composite Translation Score for Text to Java

In [ ]:
java_text_gen_results_df["translation_score"] = (
    0.35 * java_text_gen_results_df["codebert_similarity_score"] +
    0.25 * java_text_gen_results_df["ast_similarity_score"] +
    0.15 * java_text_gen_results_df["rougeL_score"] +
    0.10 * java_text_gen_results_df["rouge1_score"] +
    0.05 * java_text_gen_results_df["rouge2_score"] +
    0.10 * java_text_gen_results_df["bleu_score"]
)

print("Composite translation score calculated for Text to Java. Displaying updated DataFrame head:")
display(java_text_gen_results_df.head(10))

Composite translation score calculated for Text to Java. Displaying updated DataFrame head:


,qid,text,java_code_original,generated_java_code_first_attempt,passes_all_tests,ast_similarity_score,bleu_score,rouge1_score,rouge2_score,rougeL_score,codebert_similarity_score,translation_score
0,602,Find the first repeated character in a given s...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.6000,0.584753,0.689655,0.573427,0.675862,0.950750,0.740254
1,603,get a lucid number smaller than or equal to n.,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,False,0.3131,0.726709,0.737288,0.675214,0.737288,0.914966,0.689267
2,604,reverse words in a given string.,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.9091,0.920185,0.924242,0.846154,0.924242,0.986594,0.937970
3,605,check if the given integer is a prime number.,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.4886,0.728459,0.809524,0.709677,0.746032,0.929562,0.748684
4,606,convert degrees to radians.,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.8636,0.823912,0.903226,0.835165,0.881720,0.910040,0.881144
5,608,Find nth bell number.,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.5909,0.814611,0.873950,0.803419,0.823529,0.962794,0.817259
6,609,Find minimum possible value for the given peri...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,False,0.7500,0.616176,0.787879,0.707692,0.787879,0.944189,0.811938
7,610,Remove the k'th element from a given list.,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,False,0.7186,0.724811,0.851351,0.782313,0.837838,0.968754,0.841121
8,611,find the maximum of nth column from the given...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.9545,0.932501,0.961165,0.941176,0.961165,0.993176,0.966837
9,614,find the cumulative sum of all the values tha...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.9091,0.560110,0.783019,0.628571,0.773585,0.973954,0.849938


### Mean Composite Score for Text to Java Generation

In [ ]:
mean_translation_score_text_gen = java_text_gen_results_df['translation_score'].mean()
print(f"Mean Text to Java Translation Score: {mean_translation_score_text_gen:.4f}")

Mean Text to Java Translation Score: 0.8192


In [ ]:
java_text_gen_results_df.to_csv('text_to_java_generation_results.csv', index=False)
print("DataFrame saved to 'text_to_java_generation_results.csv'")

DataFrame saved to 'text_to_java_generation_results.csv'


### Evaluating Text to Python Generation with Pass@K

First, let's define a function to generate Python code directly from a text description.

In [ ]:
def generate_python_from_text_solution(model_to_eval, prompt_text, test_cases_str, num_solutions=1):
    full_prompt = f"""
             ### Instruction
             Generate Python code that solves the following problem:
             {prompt_text}

             ### Test cases
             {test_cases_str}

             ### Response
             """

    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)

    outputs = model_to_eval.generate(
        **inputs,
        max_new_tokens=768,
        do_sample=True,
        temperature=0.2,
        top_p=0.95,
        num_return_sequences=num_solutions,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

    return generated_texts

In [ ]:
test_cases_df = pd.read_csv('/content/java_python_translation_pairs_corrected_6.csv')
display(test_cases_df.head())

,qid,text,java_code,python_code,python_function,python_test_cases,java_test_cases,python_execution_status,java_execution_status
0,602,Find the first repeated character in a given s...,import java.io.*;\nimport java.lang.*;\nimport...,"def first_repeated_char(str1):\n for index,c ...",first_repeated_char,"[\n ""assert first_repeated_char('abcabc') == ...",public class TestRunner {\n private static ...,PASS,PASS
1,603,get a lucid number smaller than or equal to n.,import java.io.*;\nimport java.lang.*;\nimport...,def get_ludic(n):\n\tludics = []\n\tfor i in r...,get_ludic,"[\n ""assert get_ludic(10) == [1, 2, 3, 5, 7]""...",public class TestRunner {\n private static ...,PASS,PASS
2,604,reverse words in a given string.,import java.io.*;\nimport java.lang.*;\nimport...,def reverse_words(s):\n return ' '.join...,reverse_words,"[\n ""assert reverse_words('python program') =...",public class TestRunner {\n private static ...,PASS,PASS
3,605,check if the given integer is a prime number.,import java.io.*;\nimport java.lang.*;\nimport...,def prime_num(num):\n if num >=1:\n for i i...,prime_num,"[\n ""assert prime_num(13) == True"",\n ""asser...",public class TestRunner {\n private static ...,PASS,PASS
4,606,convert degrees to radians.,import java.io.*;\nimport java.lang.*;\nimport...,import math\ndef radian_degree(degree):\n radi...,radian_degree,"[\n ""assert radian_degree(90) == 1.5707963267...",public class TestRunner {\n private static ...,PASS,PASS


Next, we'll use the existing `run_mbpp_tests` function to check if the generated Python code passes the provided tests. We need to ensure the `test_cases_df` is loaded and accessible.

In [ ]:
# The test_cases_df should already be loaded from cell 2a5b6280
# If not, uncomment the line below:
# test_cases_df = pd.read_csv('/content/java_python_translation_pairs_testcases.csv')

python_text_gen_results_data = []
k = 1 # For pass@K, generate k solutions per problem

for index, row in test_cases_df.iterrows():
    problem_description = row['text']
    python_solution_original = row['python_code'] # Reference Python solution
    # python_test_cases in test_cases_df is a string representation of a list of strings
    # We need to convert it to an actual list for run_mbpp_tests
    python_test_cases_list = ast.literal_eval(row['python_test_cases'])

    # Calculate if the original Python solution passes all tests
    original_passes_all_tests = run_mbpp_tests(python_solution_original, python_test_cases_list)

    # Generate Python solutions (num_solutions = k for pass@k)
    generated_solutions_list = generate_python_from_text_solution(problem_description, row['python_test_cases'], num_solutions=k)

    all_passes_for_qid = False
    first_generated_python_code = ''

    if generated_solutions_list:
        first_generated_python_code = extract_code(generated_solutions_list[0])
        for generated_python_code_full_response in generated_solutions_list:
            current_generated_python_code = extract_code(generated_python_code_full_response)

            # Check if any of the generated solutions pass all tests
            if run_mbpp_tests(current_generated_python_code, python_test_cases_list):
                all_passes_for_qid = True
                break # Found a passing solution for this qid

    # Calculate AST similarity score between the *first* generated Python and original Python
    ast_score = 0.0
    if first_generated_python_code:
        try:
            original_python_ir = extract_python_ir(python_solution_original)
            generated_python_ir = extract_python_ir(first_generated_python_code)
            ast_score = similarity(original_python_ir, generated_python_ir)
        except Exception as e:
            print(f"Error calculating AST similarity for qid {row['qid']}: {e}")

    # Calculate other code metrics (BLEU, ROUGE, CodeBERT-like) between the *first* generated Python and original Python
    code_metrics = {
        'bleu': 0.0,
        'rouge1': 0.0,
        'rouge2': 0.0,
        'rougeL': 0.0,
        'codebert_similarity': 0.0
    }
    if first_generated_python_code:
        try:
            code_metrics = calculate_code_metrics(python_solution_original, first_generated_python_code)
        except Exception as e:
            print(f"Error calculating other metrics for qid {row['qid']}: {e}")

    python_text_gen_results_data.append({
        'qid': row['qid'],
        'text': row['text'],
        'python_code_original': python_solution_original,
        'generated_python_code_first_attempt': first_generated_python_code,
        'original_passes_all_tests': original_passes_all_tests, # Added original passes tests
        'passes_all_tests': all_passes_for_qid,
        'ast_similarity_score': ast_score,
        'bleu_score': code_metrics['bleu'],
        'rouge1_score': code_metrics['rouge1'],
        'rouge2_score': code_metrics['rouge2'],
        'rougeL_score': code_metrics['rougeL'],
        'codebert_similarity_score': code_metrics['codebert_similarity']
    })

python_text_gen_results_df = pd.DataFrame(python_text_gen_results_data)

print("Processing complete. Displaying the new DataFrame with Text-to-Python generation results:")
display(python_text_gen_results_df.head())

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

Error calculating AST similarity for qid 971: '(' was never closed (<unknown>, line 76)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Processing complete. Displaying the new DataFrame with Text-to-Python generation results:


,qid,text,python_code_original,generated_python_code_first_attempt,original_passes_all_tests,passes_all_tests,ast_similarity_score,bleu_score,rouge1_score,rouge2_score,rougeL_score,codebert_similarity_score
0,602,Find the first repeated character in a given s...,"def first_repeated_char(str1):\n for index,c ...",def first_repeated_char(string):\n for i in...,True,False,0.6364,0.182460,0.431373,0.163265,0.392157,0.866383
1,603,get a lucid number smaller than or equal to n.,def get_ludic(n):\n\tludics = []\n\tfor i in r...,"def get_ludic(n):\n ludic = [1, 2, 3, 5, 7]...",True,False,0.4394,0.168089,0.409639,0.222222,0.313253,0.891353
2,604,reverse words in a given string.,def reverse_words(s):\n return ' '.join...,def reverse_words(s):\n return ' '.join(rever...,True,True,1.0000,1.000000,1.000000,1.000000,1.000000,1.000000
3,605,check if the given integer is a prime number.,def prime_num(num):\n if num >=1:\n for i i...,def prime_num(num):\n if num > 1:\n for i ...,True,False,0.8030,0.746150,0.916667,0.869565,0.916667,0.991586
4,606,convert degrees to radians.,import math\ndef radian_degree(degree):\n radi...,def radian_degree(degree):\n radian = (degree...,True,True,1.0000,0.486498,0.750000,0.636364,0.750000,0.907696


### Pass@K and Mean Scores for Text to Python Generation

In [ ]:
pass_at_k_python = (python_text_gen_results_df['passes_all_tests'].sum() / len(python_text_gen_results_df)) * 100
print(f"Pass@{k} (Generated): {pass_at_k_python:.2f}%")

orig_pass_at_k_python = (python_text_gen_results_df['original_passes_all_tests'].sum() / len(python_text_gen_results_df)) * 100
print(f"Pass@{k} (Original): {orig_pass_at_k_python:.2f}%")

mean_ast_similarity_python_text_gen = python_text_gen_results_df['ast_similarity_score'].mean()
mean_bleu_python_text_gen = python_text_gen_results_df['bleu_score'].mean()
mean_rouge1_python_text_gen = python_text_gen_results_df['rouge1_score'].mean()
mean_rouge2_python_text_gen = python_text_gen_results_df['rouge2_score'].mean()
mean_rougeL_python_text_gen = python_text_gen_results_df['rougeL_score'].mean()
mean_codebert_similarity_python_text_gen = python_text_gen_results_df['codebert_similarity_score'].mean()

print(f"Mean AST Similarity Score (Text-to-Python): {mean_ast_similarity_python_text_gen:.4f}")
print(f"Mean BLEU Score (Text-to-Python): {mean_bleu_python_text_gen:.4f}")
print(f"Mean ROUGE-1 Score (Text-to-Python): {mean_rouge1_python_text_gen:.4f}")
print(f"Mean ROUGE-2 Score (Text-to-Python): {mean_rouge2_python_text_gen:.4f}")
print(f"Mean ROUGE-L Score (Text-to-Python): {mean_rougeL_python_text_gen:.4f}")
print(f"Mean CodeBERT Similarity Score (Text-to-Python): {mean_codebert_similarity_python_text_gen:.4f}")

Pass@1 (Generated): 53.01%
Pass@1 (Original): 100.00%
Mean AST Similarity Score (Text-to-Python): 0.7038
Mean BLEU Score (Text-to-Python): 0.2869
Mean ROUGE-1 Score (Text-to-Python): 0.5473
Mean ROUGE-2 Score (Text-to-Python): 0.3279
Mean ROUGE-L Score (Text-to-Python): 0.5014
Mean CodeBERT Similarity Score (Text-to-Python): 0.8516


### Composite Translation Score for Text to Python

In [ ]:
python_text_gen_results_df["translation_score"] = (
    0.35 * python_text_gen_results_df["codebert_similarity_score"] +
    0.25 * python_text_gen_results_df["ast_similarity_score"] +
    0.15 * python_text_gen_results_df["rougeL_score"] +
    0.10 * python_text_gen_results_df["rouge1_score"] +
    0.05 * python_text_gen_results_df["rouge2_score"] +
    0.10 * python_text_gen_results_df["bleu_score"]
)

print("Composite translation score calculated for Text to Python. Displaying updated DataFrame head:")
display(python_text_gen_results_df.head(10))

Composite translation score calculated for Text to Python. Displaying updated DataFrame head:


,qid,text,python_code_original,generated_python_code_first_attempt,original_passes_all_tests,passes_all_tests,ast_similarity_score,bleu_score,rouge1_score,rouge2_score,rougeL_score,codebert_similarity_score,translation_score
0,602,Find the first repeated character in a given s...,"def first_repeated_char(str1):\n for index,c ...",def first_repeated_char(string):\n for i in...,True,False,0.6364,0.182460,0.431373,0.163265,0.392157,0.866383,0.590704
1,603,get a lucid number smaller than or equal to n.,def get_ludic(n):\n\tludics = []\n\tfor i in r...,"def get_ludic(n):\n ludic = [1, 2, 3, 5, 7]...",True,False,0.4394,0.168089,0.409639,0.222222,0.313253,0.891353,0.537695
2,604,reverse words in a given string.,def reverse_words(s):\n return ' '.join...,def reverse_words(s):\n return ' '.join(rever...,True,True,1.0000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
3,605,check if the given integer is a prime number.,def prime_num(num):\n if num >=1:\n for i i...,def prime_num(num):\n if num > 1:\n for i ...,True,False,0.8030,0.746150,0.916667,0.869565,0.916667,0.991586,0.895065
4,606,convert degrees to radians.,import math\ndef radian_degree(degree):\n radi...,def radian_degree(degree):\n radian = (degree...,True,True,1.0000,0.486498,0.750000,0.636364,0.750000,0.907696,0.835662
5,608,Find nth bell number.,def bell_Number(n): \n bell = [[0 for i in ...,def bell_Number(n):\n if n == 0:\n r...,True,False,0.3636,0.048521,0.361446,0.222222,0.313253,0.886078,0.500123
6,609,Find minimum possible value for the given peri...,"def floor_Min(A,B,N):\n x = max(B - 1,N)\n ...","def floor_Min(a, b, c):\n return min(a, b, c)",True,False,0.7273,0.141788,0.615385,0.333333,0.615385,0.868098,0.670351
7,610,Remove the k'th element from a given list.,"def remove_kth_element(list1, L):\n return ...","def remove_kth_element(nums, k):\n nums.pop...",True,True,0.9545,0.279443,0.500000,0.272727,0.416667,0.649915,0.620176
8,611,find the maximum of nth column from the given...,"def max_of_nth(test_list, N):\n res = max([su...","def max_of_nth(l, n): \n\tres = l[0][n] \n\tfo...",True,True,0.5455,0.173562,0.478261,0.272727,0.478261,0.810407,0.570575
9,614,find the cumulative sum of all the values tha...,def cummulative_sum(test_list):\n res = sum(m...,def cummulative_sum(test_list):\n sum = 0\n ...,True,False,0.8409,0.366894,0.588235,0.375000,0.588235,0.903220,0.728850


### Mean Composite Score for Text to Python Generation

In [ ]:
mean_translation_score_python_text_gen = python_text_gen_results_df['translation_score'].mean()
print(f"Mean Text to Python Translation Score: {mean_translation_score_python_text_gen:.4f}")

Mean Text to Python Translation Score: 0.6491


In [ ]:
python_text_gen_results_df.to_csv('text_to_python_generation_results.csv', index=False)
print("DataFrame saved to 'text_to_python_generation_results.csv'")

DataFrame saved to 'text_to_python_generation_results.csv'


### Evaluating Base Model (bigcode/starcoder2-3b) for Text to Python Generation with Pass@K

Now, we'll run the Text-to-Python generation and evaluation loop using the base model. The results will be stored in a new DataFrame to distinguish them from the fine-tuned model's results.

In [ ]:
base_model_python_text_gen_results_data = []
k = 1 # For pass@K, generate k solutions per problem

for index, row in test_cases_df.iterrows():
    print(f"Processing qid: {row['qid']}")
    problem_description = row['text']
    python_solution_original = row['python_code'] # Reference Python solution
    python_test_cases_list = ast.literal_eval(row['python_test_cases'])


    generated_solutions_list = generate_python_from_text_solution_base_model(model, problem_description, row['python_test_cases'], num_solutions=k)

    all_passes_for_qid = False
    first_generated_python_code = ''

    if generated_solutions_list:
        first_generated_python_code = extract_code(generated_solutions_list[0])
        for generated_python_code_full_response in generated_solutions_list:
            current_generated_python_code = extract_code(generated_python_code_full_response)

            if run_mbpp_tests(current_generated_python_code, python_test_cases_list):
                all_passes_for_qid = True
                break

    ast_score = 0.0
    if first_generated_python_code:
        try:
            original_python_ir = extract_python_ir(python_solution_original)
            generated_python_ir = extract_python_ir(first_generated_python_code)
            ast_score = similarity(original_python_ir, generated_python_ir)
        except Exception as e:
            print(f"Error calculating AST similarity for qid {row['qid']}: {e}")

    code_metrics = {
        'bleu': 0.0,
        'rouge1': 0.0,
        'rouge2': 0.0,
        'rougeL': 0.0,
        'codebert_similarity': 0.0
    }
    if first_generated_python_code:
        try:
            code_metrics = calculate_code_metrics(python_solution_original, first_generated_python_code)
        except Exception as e:
            print(f"Error calculating other metrics for qid {row['qid']}: {e}")

    base_model_python_text_gen_results_data.append({
        'qid': row['qid'],
        'text': row['text'],
        'python_code_original': python_solution_original,
        'generated_python_code_first_attempt': first_generated_python_code,
        'passes_all_tests': all_passes_for_qid,
        'ast_similarity_score': ast_score,
        'bleu_score': code_metrics['bleu'],
        'rouge1_score': code_metrics['rouge1'],
        'rouge2_score': code_metrics['rouge2'],
        'rougeL_score': code_metrics['rougeL'],
        'codebert_similarity_score': code_metrics['codebert_similarity']
    })

base_model_python_text_gen_results_df = pd.DataFrame(base_model_python_text_gen_results_data)

print("Processing complete for Base Model. Displaying results:")
display(base_model_python_text_gen_results_df.head())

Error calculating AST similarity for qid 964: invalid decimal literal (<unknown>, line 3)
Processing qid: 965


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Error calculating AST similarity for qid 965: unindent does not match any outer indentation level (<unknown>, line 5)
Processing qid: 967


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Error calculating AST similarity for qid 967: unindent does not match any outer indentation level (<unknown>, line 3)
Processing qid: 968


...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Error calculating AST similarity for qid 973: unindent does not match any outer indentation level (<unknown>, line 5)
Processing qid: 974


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Error calculating AST similarity for qid 974: unindent does not match any outer indentation level (<unknown>, line 10)
Processing complete for Base Model. Displaying results:


,qid,text,python_code_original,generated_python_code_first_attempt,passes_all_tests,ast_similarity_score,bleu_score,rouge1_score,rouge2_score,rougeL_score,codebert_similarity_score
0,602,Find the first repeated character in a given s...,"def first_repeated_char(str1):\n for index,c ...",def first_repeated_char(s):\n ...,False,0.0,0.044140,0.167832,0.070922,0.167832,0.593619
1,603,get a lucid number smaller than or equal to n.,def get_ludic(n):\n\tludics = []\n\tfor i in r...,def get_ludic(n):\n return [i ...,False,0.0,0.092050,0.175258,0.093750,0.175258,0.648490
2,604,reverse words in a given string.,def reverse_words(s):\n return ' '.join...,def reverse_words(s):\n return...,False,0.0,0.104483,0.122449,0.110345,0.122449,0.739401
3,605,check if the given integer is a prime number.,def prime_num(num):\n if num >=1:\n for i i...,def prime_num(num):\n if num <...,False,0.0,0.094301,0.247525,0.180000,0.217822,0.690752
4,606,convert degrees to radians.,import math\ndef radian_degree(degree):\n radi...,def sum_list(numbers):\n retur...,False,0.0,0.000000,0.054054,0.000000,0.054054,0.344826


### Pass@K and Mean Scores for Base Model Text to Python Generation

In [ ]:
pass_at_k_python_base = (base_model_python_text_gen_results_df['passes_all_tests'].sum() / len(base_model_python_text_gen_results_df)) * 100
print(f"Base Model Pass@{k} (Generated): {pass_at_k_python_base:.2f}%")

mean_ast_similarity_python_text_gen_base = base_model_python_text_gen_results_df['ast_similarity_score'].mean()
mean_bleu_python_text_gen_base = base_model_python_text_gen_results_df['bleu_score'].mean()
mean_rouge1_python_text_gen_base = base_model_python_text_gen_results_df['rouge1_score'].mean()
mean_rouge2_python_text_gen_base = base_model_python_text_gen_results_df['rouge2_score'].mean()
mean_rougeL_python_text_gen_base = base_model_python_text_gen_results_df['rougeL_score'].mean()
mean_codebert_similarity_python_text_gen_base = base_model_python_text_gen_results_df['codebert_similarity_score'].mean()

print(f"Mean AST Similarity Score (Base Model Text-to-Python): {mean_ast_similarity_python_text_gen_base:.4f}")
print(f"Mean BLEU Score (Base Model Text-to-Python): {mean_bleu_python_text_gen_base:.4f}")
print(f"Mean ROUGE-1 Score (Base Model Text-to-Python): {mean_rouge1_python_text_gen_base:.4f}")
print(f"Mean ROUGE-2 Score (Base Model Text-to-Python): {mean_rouge2_python_text_gen_base:.4f}")
print(f"Mean ROUGE-L Score (Base Model Text-to-Python): {mean_rougeL_python_text_gen_base:.4f}")
print(f"Mean CodeBERT Similarity Score (Base Model Text-to-Python): {mean_codebert_similarity_python_text_gen_base:.4f}")

Base Model Pass@1 (Generated): 0.60%
Mean AST Similarity Score (Base Model Text-to-Python): 0.0123
Mean BLEU Score (Base Model Text-to-Python): 0.0557
Mean ROUGE-1 Score (Base Model Text-to-Python): 0.1909
Mean ROUGE-2 Score (Base Model Text-to-Python): 0.0839
Mean ROUGE-L Score (Base Model Text-to-Python): 0.1701
Mean CodeBERT Similarity Score (Base Model Text-to-Python): 0.6773


### Composite Translation Score for Base Model Text to Python

In [ ]:
base_model_python_text_gen_results_df["translation_score"] = (
    0.35 * base_model_python_text_gen_results_df["codebert_similarity_score"] +
    0.25 * base_model_python_text_gen_results_df["ast_similarity_score"] +
    0.15 * base_model_python_text_gen_results_df["rougeL_score"] +
    0.10 * base_model_python_text_gen_results_df["rouge1_score"] +
    0.05 * base_model_python_text_gen_results_df["rouge2_score"] +
    0.10 * base_model_python_text_gen_results_df["bleu_score"]
)

print("Composite translation score calculated for Base Model Text to Python. Displaying updated DataFrame head:")
display(base_model_python_text_gen_results_df.head(10))

Composite translation score calculated for Base Model Text to Python. Displaying updated DataFrame head:


,qid,text,python_code_original,generated_python_code_first_attempt,passes_all_tests,ast_similarity_score,bleu_score,rouge1_score,rouge2_score,rougeL_score,codebert_similarity_score,translation_score
0,602,Find the first repeated character in a given s...,"def first_repeated_char(str1):\n for index,c ...",def first_repeated_char(s):\n ...,False,0.0,0.044140,0.167832,0.070922,0.167832,0.593619,0.257685
1,603,get a lucid number smaller than or equal to n.,def get_ludic(n):\n\tludics = []\n\tfor i in r...,def get_ludic(n):\n return [i ...,False,0.0,0.092050,0.175258,0.093750,0.175258,0.648490,0.284679
2,604,reverse words in a given string.,def reverse_words(s):\n return ' '.join...,def reverse_words(s):\n return...,False,0.0,0.104483,0.122449,0.110345,0.122449,0.739401,0.305368
3,605,check if the given integer is a prime number.,def prime_num(num):\n if num >=1:\n for i i...,def prime_num(num):\n if num <...,False,0.0,0.094301,0.247525,0.180000,0.217822,0.690752,0.317619
4,606,convert degrees to radians.,import math\ndef radian_degree(degree):\n radi...,def sum_list(numbers):\n retur...,False,0.0,0.000000,0.054054,0.000000,0.054054,0.344826,0.134203
5,608,Find nth bell number.,def bell_Number(n): \n bell = [[0 for i in ...,def bell_Number(n):\n return 0...,False,0.0,0.053715,0.331492,0.122905,0.287293,0.724176,0.341222
6,609,Find minimum possible value for the given peri...,"def floor_Min(A,B,N):\n x = max(B - 1,N)\n ...","def floor_Min(a, b, c):\n retu...",False,0.0,0.019777,0.158730,0.064516,0.158730,0.655806,0.274418
7,610,Remove the k'th element from a given list.,"def remove_kth_element(list1, L):\n return ...","def remove_kth_element(lst, k):\n ...",False,0.0,0.036657,0.103448,0.052632,0.103448,0.719701,0.284055
8,611,find the maximum of nth column from the given...,"def max_of_nth(test_list, N):\n res = max([su...","def max_of_nth(tuple_list, n):\n ...",False,0.0,0.035157,0.193103,0.069930,0.165517,0.742963,0.311187
9,614,find the cumulative sum of all the values tha...,def cummulative_sum(test_list):\n res = sum(m...,def cummulative_sum(lst):\n re...,False,0.0,0.021233,0.132450,0.026846,0.105960,0.647128,0.259099


### Mean Composite Score for Base Model Text to Python Generation

In [ ]:
mean_translation_score_python_text_gen_base = base_model_python_text_gen_results_df['translation_score'].mean()
print(f"Mean Base Model Text to Python Translation Score: {mean_translation_score_python_text_gen_base:.4f}")

Mean Base Model Text to Python Translation Score: 0.2945


In [ ]:
base_model_python_text_gen_results_df.to_csv('base_model_text_to_python_generation_results.csv', index=False)
print("DataFrame saved to 'base_model_text_to_python_generation_results.csv'")

DataFrame saved to 'base_model_text_to_python_generation_results.csv'


### Evaluating Base Model (bigcode/starcoder2-3b) for Text to Java Generation with Pass@K

Now, we'll run the Text-to-Java generation and evaluation loop using the base model. The results will be stored in a new DataFrame to distinguish them from the fine-tuned model's results.

In [ ]:
def generate_java_from_text_solution_base_model(model_eval, prompt_text, test_cases, num_solutions=1):
    full_prompt = f"""
             ### Instruction
             Generate Java code that solves the following problem:
             {prompt_text}

             ### Test cases
             {test_cases}

             ### Response
             """

    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)

    outputs = model_eval.generate(
        **inputs,
        max_new_tokens=768,
        do_sample=True,
        temperature=0.2,
        top_p=0.95,
        num_return_sequences=num_solutions,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

    return generated_texts

In [ ]:
base_model_java_text_gen_results_data = []
k = 1 # For pass@1

for index, row in test_cases_df.iterrows():
    print(f"Processing qid: {row['qid']}")
    problem_description = row['text']
    java_solution_original = row['java_code'] # Reference Java solution
    test_cases_str = row['java_test_cases']

    # Check if original Java solution passes all tests
    original_passes_all_tests = run_java_tests(java_solution_original, test_cases_str)

    # Generate Java solutions (num_solutions = k for pass@k) using the base model
    generated_solutions_list = generate_java_from_text_solution_base_model(model, problem_description, test_cases_str, num_solutions=k)

    all_passes_for_qid = False
    first_generated_java_code = ''

    if generated_solutions_list:
        first_generated_java_code = extract_code(generated_solutions_list[0])
        for generated_java_code_full_response in generated_solutions_list:
            current_generated_java_code = extract_code(generated_java_code_full_response)

            # Check if any of the generated solutions pass all tests using the provided test runner code
            if run_java_tests(current_generated_java_code, test_cases_str):
                all_passes_for_qid = True
                break # Found a passing solution for this qid

    # Calculate AST similarity score between the *first* generated Java and original Java
    ast_score = 0.0
    if generated_solutions_list:
        try:
            # first_generated_java_code is already extracted above
            ast_score = compare_java_java(java_solution_original, first_generated_java_code)
        except Exception as e:
            print(f"Error calculating AST similarity for qid {row['qid']}: {e}")

    # Calculate other code metrics (BLEU, ROUGE, CodeBERT-like) between the *first* generated Java and original Java
    code_metrics = {
        'bleu': 0.0,
        'rouge1': 0.0,
        'rouge2': 0.0,
        'rougeL': 0.0,
        'codebert_similarity': 0.0
    }
    if generated_solutions_list:
        try:
            # first_generated_java_code is already extracted above
            code_metrics = calculate_code_metrics(java_solution_original, first_generated_java_code)
        except Exception as e:
            print(f"Error calculating other metrics for qid {row['qid']}: {e}")


    base_model_java_text_gen_results_data.append({
        'qid': row['qid'],
        'text': row['text'],
        'java_code_original': java_solution_original,
        'generated_java_code_first_attempt': first_generated_java_code if generated_solutions_list else '',
        'original_passes_all_tests': original_passes_all_tests,
        'passes_all_tests': all_passes_for_qid,
        'ast_similarity_score': ast_score,
        'bleu_score': code_metrics['bleu'],
        'rouge1_score': code_metrics['rouge1'],
        'rouge2_score': code_metrics['rouge2'],
        'rougeL_score': code_metrics['rougeL'],
        'codebert_similarity_score': code_metrics['codebert_similarity']
    })

base_model_java_text_gen_results_df = pd.DataFrame(base_model_java_text_gen_results_data)

print("Processing complete for Base Model Text-to-Java generation. Displaying results:")
display(base_model_java_text_gen_results_df.head())

Processing qid: 629


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Processing qid: 630


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Processing qid: 631


...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Processing qid: 645


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Processing qid: 646


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


### Pass@K and Mean Scores for Base Model Text to Java Generation

In [ ]:
pass_at_k_java_base = (base_model_java_text_gen_results_df['passes_all_tests'].sum() / len(base_model_java_text_gen_results_df)) * 100
print(f"Base Model Pass@{k} (Generated): {pass_at_k_java_base:.2f}%")

orig_pass_at_k_java_base = (base_model_java_text_gen_results_df['original_passes_all_tests'].sum() / len(base_model_java_text_gen_results_df)) * 100
print(f"Base Model Pass@{k} (Original): {orig_pass_at_k_java_base:.2f}%")

mean_ast_similarity_java_text_gen_base = base_model_java_text_gen_results_df['ast_similarity_score'].mean()
mean_bleu_java_text_gen_base = base_model_java_text_gen_results_df['bleu_score'].mean()
mean_rouge1_java_text_gen_base = base_model_java_text_gen_results_df['rouge1_score'].mean()
mean_rouge2_java_text_gen_base = base_model_java_text_gen_results_df['rouge2_score'].mean()
mean_rougeL_java_text_gen_base = base_model_java_text_gen_results_df['rougeL_score'].mean()
mean_codebert_similarity_java_text_gen_base = base_model_java_text_gen_results_df['codebert_similarity_score'].mean()

print(f"Mean AST Similarity Score (Base Model Text-to-Java): {mean_ast_similarity_java_text_gen_base:.4f}")
print(f"Mean BLEU Score (Base Model Text-to-Java): {mean_bleu_java_text_gen_base:.4f}")
print(f"Mean ROUGE-1 Score (Base Model Text-to-Java): {mean_rouge1_java_text_gen_base:.4f}")
print(f"Mean ROUGE-2 Score (Base Model Text-to-Java): {mean_rouge2_java_text_gen_base:.4f}")
print(f"Mean ROUGE-L Score (Base Model Text-to-Java): {mean_rougeL_java_text_gen_base:.4f}")
print(f"Mean CodeBERT Similarity Score (Base Model Text-to-Java): {mean_codebert_similarity_java_text_gen_base:.4f}")

Base Model Pass@1 (Generated): 0.00%
Base Model Pass@1 (Original): 100.00%
Mean AST Similarity Score (Base Model Text-to-Java): 0.5415
Mean BLEU Score (Base Model Text-to-Java): 0.0634
Mean ROUGE-1 Score (Base Model Text-to-Java): 0.1834
Mean ROUGE-2 Score (Base Model Text-to-Java): 0.0939
Mean ROUGE-L Score (Base Model Text-to-Java): 0.1268
Mean CodeBERT Similarity Score (Base Model Text-to-Java): 0.6234


### Composite Translation Score for Base Model Text to Java

In [ ]:
base_model_java_text_gen_results_df["translation_score"] = (
    0.35 * base_model_java_text_gen_results_df["codebert_similarity_score"] +
    0.25 * base_model_java_text_gen_results_df["ast_similarity_score"] +
    0.15 * base_model_java_text_gen_results_df["rougeL_score"] +
    0.10 * base_model_java_text_gen_results_df["rouge1_score"] +
    0.05 * base_model_java_text_gen_results_df["rouge2_score"] +
    0.10 * base_model_java_text_gen_results_df["bleu_score"]
)

print("Composite translation score calculated for Base Model Text to Java. Displaying updated DataFrame head:")
display(base_model_java_text_gen_results_df.head(10))

Composite translation score calculated for Base Model Text to Java. Displaying updated DataFrame head:


,qid,text,java_code_original,generated_java_code_first_attempt,original_passes_all_tests,passes_all_tests,ast_similarity_score,bleu_score,rouge1_score,rouge2_score,rougeL_score,codebert_similarity_score,translation_score
0,602,Find the first repeated character in a given s...,import java.io.*;\nimport java.lang.*;\nimport...,public class FirstRepeatedChar {\n public s...,True,False,0.3939,0.175140,0.341014,0.186047,0.221198,0.540645,0.381798
1,603,get a lucid number smaller than or equal to n.,import java.io.*;\nimport java.lang.*;\nimport...,public class GetLudic {\n public static jav...,True,False,0.5844,0.140046,0.222951,0.138614,0.124590,0.808261,0.490910
2,604,reverse words in a given string.,import java.io.*;\nimport java.lang.*;\nimport...,public class ReverseWords {\n public static...,True,False,0.3374,0.106604,0.243781,0.160000,0.184080,0.671354,0.389974
3,605,check if the given integer is a prime number.,import java.io.*;\nimport java.lang.*;\nimport...,public class PrimeNum {\n public static boo...,True,False,0.3636,0.012196,0.275000,0.179487,0.250000,0.743734,0.426401
4,606,convert degrees to radians.,import java.io.*;\nimport java.lang.*;\nimport...,public class RadianDegree {\n public static...,True,False,0.5758,0.032655,0.207254,0.052356,0.113990,0.648167,0.414516
5,608,Find nth bell number.,import java.io.*;\nimport java.lang.*;\nimport...,public class BellNumber {\n public static i...,True,False,0.3939,0.046336,0.175573,0.092308,0.145038,0.541694,0.336630
6,609,Find minimum possible value for the given peri...,import java.io.*;\nimport java.lang.*;\nimport...,public class FloorMin {\n public static int...,True,False,0.7273,0.027812,0.077739,0.035461,0.060071,0.340549,0.322356
7,610,Remove the k'th element from a given list.,import java.io.*;\nimport java.lang.*;\nimport...,public class RemoveKthElement {\n public st...,True,False,0.5844,0.026300,0.088657,0.028758,0.059974,0.789178,0.444242
8,611,find the maximum of nth column from the given...,import java.io.*;\nimport java.lang.*;\nimport...,public class MaxOfNth {\n public static int...,True,False,0.5909,0.034297,0.134940,0.043584,0.057831,0.606925,0.387926
9,614,find the cumulative sum of all the values tha...,import java.io.*;\nimport java.lang.*;\nimport...,public class CummulativeSum {\n public stat...,True,False,0.8636,0.032334,0.066667,0.022293,0.050794,0.543588,0.424790


### Mean Composite Score for Base Model Text to Java Generation

In [ ]:
mean_translation_score_java_text_gen_base = base_model_java_text_gen_results_df['translation_score'].mean()
print(f"Mean Base Model Text to Java Translation Score: {mean_translation_score_java_text_gen_base:.4f}")

Mean Base Model Text to Java Translation Score: 0.4020


In [ ]:
base_model_java_text_gen_results_df.to_csv('base_model_text_to_java_generation_results.csv', index=False)
print("DataFrame saved to 'base_model_text_to_java_generation_results.csv'")

DataFrame saved to 'base_model_text_to_java_generation_results.csv'


### Evaluating Base Model (bigcode/starcoder2-3b) for Python to Java Translation

Now, we'll define a function to generate Java code from Python using the base model, and then run the evaluation loop for Python to Java translation, calculate all specified metrics, and store them in a new DataFrame.

In [ ]:
def generate_java_from_python_solution_base_model(model_eval, python_code_problem, num_solutions=1):
    full_prompt = f"""
             ### Instruction
             Convert the following Python code to Java:
             {python_code_problem}

             ### Response
             """

    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)

    outputs = model_eval.generate(
        **inputs,
        max_new_tokens=768,
        do_sample=True,
        temperature=0.2,
        top_p=0.95,
        num_return_sequences=num_solutions,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

    return generated_texts

In [ ]:
base_model_python_to_java_results_data = []
k=1 # For Pass@k

for index, row in test_cases_df[300:].iterrows():
    print(f"Processing qid: {row['qid']}")
    python_problem = row['python_code']
    java_solution_original = row['java_code'] # Reference Java solution
    test_cases_str = row['java_test_cases'] # Extract test cases

    # Check if original Java solution compiles (still useful to know)
    original_compiles = check_java_compilation(java_solution_original)
    # Check if original Java solution passes all tests
    original_passes_all_tests = run_java_tests(java_solution_original, test_cases_str)

    # Generate Java solutions (num_solutions = k for pass@k) using the base model
    generated_solutions_list = generate_java_from_python_solution_base_model(model, python_problem, num_solutions=k)

    passes_all_tests = False # This will be our pass@k
    compiles_successfully = False # Track if any generated solution compiles
    first_generated_java_code = ''

    if generated_solutions_list:
        first_generated_java_code = extract_code(generated_solutions_list[0])
        for generated_java_code_full_response in generated_solutions_list:
            current_generated_java_code = extract_code(generated_java_code_full_response)
            # Check if any of the generated solutions compile
            if check_java_compilation(current_generated_java_code):
                compiles_successfully = True
            # Check if any of the generated solutions pass all tests
            if run_java_tests(current_generated_java_code, test_cases_str):
                passes_all_tests = True

            if compiles_successfully and passes_all_tests: # If both conditions met, no need to check further for this qid
                break

    # Calculate AST similarity score between the *first* generated Java and original Java
    ast_score = 0.0
    if generated_solutions_list:
        try:
            # first_generated_java_code is already extracted above
            ast_score = compare_java_java(java_solution_original, first_generated_java_code)
        except Exception as e:
            print(f"Error calculating AST similarity for qid {row['qid']}: {e}")

    # Calculate other code metrics (BLEU, ROUGE, CodeBERT-like) between the *first* generated Java and original Java
    code_metrics = {
        'bleu': 0.0,
        'rouge1': 0.0,
        'rouge2': 0.0,
        'rougeL': 0.0,
        'codebert_similarity': 0.0
    }
    if generated_solutions_list:
        try:
            # first_generated_java_code is already extracted above
            code_metrics = calculate_code_metrics(java_solution_original, first_generated_java_code)
        except Exception as e:
            print(f"Error calculating other metrics for qid {row['qid']}: {e}")


    base_model_python_to_java_results_data.append({
        'qid': row['qid'],
        'text': row['text'],
        'python_code_original': python_problem,
        'java_code_original': java_solution_original,
        'generated_java_code_first_attempt': first_generated_java_code if generated_solutions_list else '',
        'original_compiles': original_compiles,
        'original_passes_all_tests': original_passes_all_tests, # New field
        'compiles_successfully': compiles_successfully, # Re-added field
        'passes_all_tests': passes_all_tests, # Renamed and updated logic
        'ast_similarity_score': ast_score,
        'bleu_score': code_metrics['bleu'],
        'rouge1_score': code_metrics['rouge1'],
        'rouge2_score': code_metrics['rouge2'],
        'rougeL_score': code_metrics['rougeL'],
        'codebert_similarity_score': code_metrics['codebert_similarity']
    })

base_model_python_to_java_results_df = pd.DataFrame(base_model_python_to_java_results_data)

print("Processing complete for Base Model Python-to-Java translation. Displaying results:")
display(base_model_python_to_java_results_df.head())

Processing qid: 956


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Processing qid: 957


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Processing qid: 958


...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Processing qid: 974


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Processing complete for Base Model Python-to-Java translation. Displaying results:


,qid,text,python_code_original,java_code_original,generated_java_code_first_attempt,original_compiles,original_passes_all_tests,compiles_successfully,passes_all_tests,ast_similarity_score,bleu_score,rouge1_score,rouge2_score,rougeL_score,codebert_similarity_score
0,937,count the most common character in a given st...,from collections import Counter \ndef max_char...,import java.io.*;\nimport java.lang.*;\nimport...,public static String reverseString,True,True,False,False,0.3636,0.000000,0.055046,0.037383,0.055046,0.242506
1,938,find three closest elements from three sorted...,"import sys \n\ndef find_closet(A, B, C, p, q, ...",import java.io.*;\nimport java.lang.*;\nimport...,"public static int[] findCloset(int[] A, int[] ...",True,True,False,False,0.2675,0.260317,0.455408,0.289524,0.409867,0.685654
2,940,sort the given list by using heap sort.,def heap_sort(arr):\n heapify(arr) \n e...,import java.io.*;\nimport java.lang.*;\nimport...,public static int[] heapSort(int[] arr) {\n ...,True,True,False,False,0.3030,0.042139,0.154206,0.023474,0.070093,0.686643
3,942,check if any list element is present in the g...,"def check_element(test_tup, check_list):\n re...",import java.io.*;\nimport java.lang.*;\nimport...,"public boolean checkElement(Tuple testTup, Lis...",True,True,False,False,0.3182,0.039171,0.189974,0.021220,0.142480,0.732335
4,943,combine two given sorted lists using heapq mo...,from heapq import merge\ndef combine_lists(num...,import java.io.*;\nimport java.lang.*;\nimport...,public static List<Integer> combineLists(List<...,True,True,False,False,0.6970,0.007706,0.269006,0.142012,0.198830,0.733323


### Compilation Rate and Mean Scores for Base Model Python to Java Translation

In [ ]:
pass_at_k_python_to_java_base = (base_model_python_to_java_results_df['passes_all_tests'].sum() / len(base_model_python_to_java_results_df)) * 100
print(f"Base Model Pass@{k} (Generated Java): {pass_at_k_python_to_java_base:.2f}%")

original_pass_at_k_python_to_java_base = (base_model_python_to_java_results_df['original_passes_all_tests'].sum() / len(base_model_python_to_java_results_df)) * 100
print(f"Base Model Pass@{k} (Original Java): {original_pass_at_k_python_to_java_base:.2f}%")

compilation_rate_base = (base_model_python_to_java_results_df['compiles_successfully'].sum() / len(base_model_python_to_java_results_df)) * 100
print(f"Base Model Compilation Rate (Generated Java): {compilation_rate_base:.2f}%")

original_compilation_rate_base = (base_model_python_to_java_results_df['original_compiles'].sum() / len(base_model_python_to_java_results_df)) * 100
print(f"Base Model Compilation Rate (Original Java): {original_compilation_rate_base:.2f}%")

mean_ast_similarity_python_to_java_base = base_model_python_to_java_results_df['ast_similarity_score'].mean()
mean_bleu_python_to_java_base = base_model_python_to_java_results_df['bleu_score'].mean()
mean_rouge1_python_to_java_base = base_model_python_to_java_results_df['rouge1_score'].mean()
mean_rouge2_python_to_java_results_df = base_model_python_to_java_results_df['rouge2_score'].mean()
mean_rougeL_python_to_java_base = base_model_python_to_java_results_df['rougeL_score'].mean()
mean_codebert_similarity_python_to_java_base = base_model_python_to_java_results_df['codebert_similarity_score'].mean()

print(f"Mean AST Similarity Score (Base Model Python-to-Java): {mean_ast_similarity_python_to_java_base:.4f}")
print(f"Mean BLEU Score (Base Model Python-to-Java): {mean_bleu_python_to_java_base:.4f}")
print(f"Mean ROUGE-1 Score (Base Model Python-to-Java): {mean_rouge1_python_to_java_base:.4f}")
print(f"Mean ROUGE-2 Score (Base Model Python-to-Java): {mean_rouge2_python_to_java_results_df:.4f}")
print(f"Mean ROUGE-L Score (Base Model Python-to-Java): {mean_rougeL_python_to_java_base:.4f}")
print(f"Mean CodeBERT Similarity Score (Base Model Python-to-Java): {mean_codebert_similarity_python_to_java_base:.4f}")

NameError: name 'base_model_python_to_java_results_df' is not defined

### Composite Translation Score for Base Model Python to Java

In [ ]:
base_model_python_to_java_results_df["translation_score"] = (
    0.35 * base_model_python_to_java_results_df["codebert_similarity_score"] +
    0.25 * base_model_python_to_java_results_df["ast_similarity_score"] +
    0.15 * base_model_python_to_java_results_df["rougeL_score"] +
    0.10 * base_model_python_to_java_results_df["rouge1_score"] +
    0.05 * base_model_python_to_java_results_df["rouge2_score"] +
    0.10 * base_model_python_to_java_results_df["bleu_score"]
)

print("Composite translation score calculated for Base Model Python to Java. Displaying updated DataFrame head:")
display(base_model_python_to_java_results_df.head(10))

Composite translation score calculated for Base Model Python to Java. Displaying updated DataFrame head:


,qid,text,python_code_original,java_code_original,generated_java_code_first_attempt,original_compiles,original_passes_all_tests,compiles_successfully,passes_all_tests,ast_similarity_score,bleu_score,rouge1_score,rouge2_score,rougeL_score,codebert_similarity_score,translation_score
0,937,count the most common character in a given st...,from collections import Counter \ndef max_char...,import java.io.*;\nimport java.lang.*;\nimport...,public static String reverseString,True,True,False,False,0.3636,0.000000,0.055046,0.037383,0.055046,0.242506,0.191408
1,938,find three closest elements from three sorted...,"import sys \n\ndef find_closet(A, B, C, p, q, ...",import java.io.*;\nimport java.lang.*;\nimport...,"public static int[] findCloset(int[] A, int[] ...",True,True,False,False,0.2675,0.260317,0.455408,0.289524,0.409867,0.685654,0.454383
2,940,sort the given list by using heap sort.,def heap_sort(arr):\n heapify(arr) \n e...,import java.io.*;\nimport java.lang.*;\nimport...,public static int[] heapSort(int[] arr) {\n ...,True,True,False,False,0.3030,0.042139,0.154206,0.023474,0.070093,0.686643,0.347397
3,942,check if any list element is present in the g...,"def check_element(test_tup, check_list):\n re...",import java.io.*;\nimport java.lang.*;\nimport...,"public boolean checkElement(Tuple testTup, Lis...",True,True,False,False,0.3182,0.039171,0.189974,0.021220,0.142480,0.732335,0.381215
4,943,combine two given sorted lists using heapq mo...,from heapq import merge\ndef combine_lists(num...,import java.io.*;\nimport java.lang.*;\nimport...,public static List<Integer> combineLists(List<...,True,True,False,False,0.6970,0.007706,0.269006,0.142012,0.198830,0.733323,0.495509
5,944,separate and print the numbers and their posi...,import re\ndef num_position(text):\n for m in ...,import java.io.*;\nimport java.lang.*;\nimport...,import re\n\ndef num_position(text):\n for ...,True,True,False,False,0.4545,0.000000,0.041131,0.005168,0.030848,0.523382,0.305807
6,945,convert the given tuples into set.,def tuple_to_set(t):\n s = set(t)\n return (s),import java.io.*;\nimport java.lang.*;\nimport...,public static Set<Integer> tuple_to_set(int[] ...,True,False,False,False,0.8636,0.024862,0.103093,0.015544,0.051546,0.679634,0.475077
7,947,Find the length of the shortest word.,def len_log(list1):\n min=len(list1[0])\n ...,import java.io.*;\nimport java.lang.*;\nimport...,public static int len_log(String[] list1) {\n ...,True,True,False,False,0.4727,0.029489,0.124069,0.019950,0.064516,0.576312,0.345915
8,949,sort the given tuple list basis the total dig...,def count_digs(tup):\n return sum([len(str(el...,import java.io.*;\nimport java.lang.*;\nimport...,public static int count_digs(int[] tup) {\n ...,True,True,False,False,0.2165,0.170509,0.333333,0.114537,0.127193,0.591916,0.336486
9,950,display sign of the chinese zodiac for given ...,def chinese_zodiac(year):\n if (year - 2000) %...,import java.io.*;\nimport java.lang.*;\nimport...,public String chineseZodiac(int year) {\n ...,True,True,False,False,0.4773,0.070704,0.155620,0.063768,0.097983,0.702794,0.405821


### Mean Composite Score for Base Model Python to Java Translation

In [ ]:
mean_translation_score_python_to_java_base = base_model_python_to_java_results_df['translation_score'].mean()
print(f"Mean Base Model Python to Java Translation Score: {mean_translation_score_python_to_java_base:.4f}")

Mean Base Model Python to Java Translation Score: 0.4128


In [ ]:
base_model_python_to_java_results_df.to_csv('base_model_python_to_java_translation_results.csv', index=False)
print("DataFrame saved to 'base_model_python_to_java_translation_results.csv'")

DataFrame saved to 'base_model_python_to_java_translation_results.csv'


Finetuned Java to python

### Detailed Metrics for Fine-tuned Model (Java to Python Translation)

In [ ]:
base_model_python_to_java_results_data = []
k=1 # For Pass@k

for index, row in test_cases_df[300:].iterrows():
    print(f"Processing qid: {row['qid']}")
    python_problem = row['python_code']
    java_solution_original = row['java_code'] # Reference Java solution
    test_cases_str = row['java_test_cases'] # Extract test cases

    # Check if original Java solution compiles (still useful to know)
    original_compiles = check_java_compilation(java_solution_original)
    # Check if original Java solution passes all tests
    original_passes_all_tests = run_java_tests(java_solution_original, test_cases_str)

    # Generate Java solutions (num_solutions = k for pass@k) using the base model
    generated_solutions_list = generate_python_from_java_solution_finetuned_model(model, python_problem, num_solutions=k)

    passes_all_tests = False # This will be our pass@k
    compiles_successfully = False # Track if any generated solution compiles
    first_generated_java_code = ''

    if generated_solutions_list:
        first_generated_java_code = extract_code(generated_solutions_list[0])
        for generated_java_code_full_response in generated_solutions_list:
            current_generated_java_code = extract_code(generated_java_code_full_response)
            # Check if any of the generated solutions compile
            if check_java_compilation(current_generated_java_code):
                compiles_successfully = True
            # Check if any of the generated solutions pass all tests
            if run_java_tests(current_generated_java_code, test_cases_str):
                passes_all_tests = True

            if compiles_successfully and passes_all_tests: # If both conditions met, no need to check further for this qid
                break

    # Calculate AST similarity score between the *first* generated Java and original Java
    ast_score = 0.0
    if generated_solutions_list:
        try:
            # first_generated_java_code is already extracted above
            ast_score = compare_java_java(java_solution_original, first_generated_java_code)
        except Exception as e:
            print(f"Error calculating AST similarity for qid {row['qid']}: {e}")

    # Calculate other code metrics (BLEU, ROUGE, CodeBERT-like) between the *first* generated Java and original Java
    code_metrics = {
        'bleu': 0.0,
        'rouge1': 0.0,
        'rouge2': 0.0,
        'rougeL': 0.0,
        'codebert_similarity': 0.0
    }
    if generated_solutions_list:
        try:
            # first_generated_java_code is already extracted above
            code_metrics = calculate_code_metrics(java_solution_original, first_generated_java_code)
        except Exception as e:
            print(f"Error calculating other metrics for qid {row['qid']}: {e}")


    base_model_python_to_java_results_data.append({
        'qid': row['qid'],
        'text': row['text'],
        'python_code_original': python_problem,
        'java_code_original': java_solution_original,
        'generated_java_code_first_attempt': first_generated_java_code if generated_solutions_list else '',
        'original_compiles': original_compiles,
        'original_passes_all_tests': original_passes_all_tests, # New field
        'compiles_successfully': compiles_successfully, # Re-added field
        'passes_all_tests': passes_all_tests, # Renamed and updated logic
        'ast_similarity_score': ast_score,
        'bleu_score': code_metrics['bleu'],
        'rouge1_score': code_metrics['rouge1'],
        'rouge2_score': code_metrics['rouge2'],
        'rougeL_score': code_metrics['rougeL'],
        'codebert_similarity_score': code_metrics['codebert_similarity']
    })

base_model_python_to_java_results_df = pd.DataFrame(base_model_python_to_java_results_data)

print("Processing complete for Base Model Python-to-Java translation. Displaying results:")
display(base_model_python_to_java_results_df.head())

In [ ]:
k = 1 # Define k for pass@k if not already defined
pass_at_k_java_to_python_finetuned = (results_df['passes_all_tests'].sum() / len(results_df)) * 100
print(f"Fine-tuned Model Pass@{k} for Java to Python: {pass_at_k_java_to_python_finetuned:.2f}%")

In [ ]:
mean_ast_similarity_finetuned = results_df['ast_similarity_score'].mean()
mean_bleu_finetuned = results_df['bleu_score'].mean()
mean_rouge1_finetuned = results_df['rouge1_score'].mean()
mean_rouge2_finetuned = results_df['rouge2_score'].mean()
mean_rougeL_finetuned = results_df['rougeL_score'].mean()
mean_codebert_similarity_finetuned = results_df['codebert_similarity_score'].mean()

print(f"Mean AST Similarity Score (Fine-tuned Java-to-Python): {mean_ast_similarity_finetuned:.4f}")
print(f"Mean BLEU Score (Fine-tuned Java-to-Python): {mean_bleu_finetuned:.4f}")
print(f"Mean ROUGE-1 Score (Fine-tuned Java-to-Python): {mean_rouge1_finetuned:.4f}")
print(f"Mean ROUGE-2 Score (Fine-tuned Java-to-Python): {mean_rouge2_finetuned:.4f}")
print(f"Mean ROUGE-L Score (Fine-tuned Java-to-Python): {mean_rougeL_finetuned:.4f}")
print(f"Mean CodeBERT Similarity Score (Fine-tuned Java-to-Python): {mean_codebert_similarity_finetuned:.4f}")

In [ ]:
results_df["translation_score"] = (
    0.35 * results_df["codebert_similarity_score"] +
    0.25 * results_df["ast_similarity_score"] +
    0.15 * results_df["rougeL_score"] +
    0.10 * results_df["rouge1_score"] +
    0.05 * results_df["rouge2_score"] +
    0.10 * results_df["bleu_score"]
)

mean_translation_score_finetuned = results_df['translation_score'].mean()
print(f"Mean Composite Translation Score (Fine-tuned Java-to-Python): {mean_translation_score_finetuned:.4f}")

display(results_df.head())